**<h1><center>Ingeniería de Datos de <font color='Purple'>Restaurantes en San Luis Potosí</font></h1></center>**

**<h3><center><font color='Plum'>Aura De La Garza García</font></h2></center>**

----------------------------------------------------------------------------------------------------------

# Introducción

Los datos proporcionados registran información sobre restaurantes y sus diferentes 
dimensiones: horarios, tipo de cocina, métodos de pago y estacionamientos. Adicionalmente se 
registra información de usuarios de los mismos restaurantes, así como tipo de cocina predilecta 
y tipos de pago. Ambas entidades se relacionan a través de las calificaciones que el usuario da 
al restaurante.  

# Objetivo 

**1.** Comenzaremos por Extraer los datos de los archivos CSV y Excel. Luego, seleccionaremos una unidad muestral de nuestros datos. En este caso, nos enfocaremos en seleccionar los restaurantes de la ciudad de ***San Luis Potosí***, así como los usuarios de estos restaurantes. Debemos comprobar que esta unidad muestral represente más del 10% de los datos totales. 

**2.** Una vez que extraimos la unidad muestral, necesitamos crear nuevas variables que nos permitan analizar el desempeño de los restaurantes en San Luis Potosí. También creamos nuestra variable objetivo. 

**3.** Hacemos un análisis de tratamiento de outliers a partir de métodos tanto univariados como multivariados. Decidimos tratar los outliers por el método que menos reduzca nuestra unidad muestral. 

**4.** Utilizando Clustering de variables, removemos multicolinealidad en variables. Luego con el método de PCA, reducimos nuestro conjunto de datos a 2 dimensiones, realizamos una visualización del conjunto. 

**5.** Seleccionamos las mejores variables predictoras utilizando WoE y IV. 

**6.** Obtenemos conclusiones sobre los datos resultantes.


# 1. Extracción de Datos

Comenzamos por cargar las librerias necesarias para la lectura de datos

In [1]:
# Importamos librerias para lectura de archivos CSV y Excel
import pandas as pd
import numpy as np

# Importamos libreria para manejar rutas de archivos
import glob
import os

Luego, continuamos por definir las rutas para nuestros datos. Podemos notar que existen dos tipos de formatos para nuestros archivos: CSV y Excel. Por lo tanto, definimos las rutas para cada uno de ellos.  

In [2]:
# Definimos la ruta de los archivos CSV y Excel
ruta_datos_csv = 'datos/*.csv'
ruta_datos_excel = 'datos/*.xlsx'

# Seleccionamos todos los archivos CSV y Excel en la ruta especificada
archivos_csv = glob.glob(ruta_datos_csv)
archivos_excel = glob.glob(ruta_datos_excel)

## 1.1 Archivos CSV

Para una extracción de datos más eficaz, optamos por cargar nuestros archivos CSV a un diccionario. En este diccionario, definimos la llave de cada CSV por su nombre. Con ello, podemos acceder a cada dataframe a traves de este diccionario. Una vez definido el diccionario, procedemos a comprobar que las 9 tablas necesarias han sido cargadas.

In [3]:
# Cargamos todos los archivos CSV en un diccionario de DataFrames
dataframes_csv = {}

print("Archivos CSV cargados:")
for ruta in archivos_csv:
    # Extraer el nombre del archivo, sin extensión
    nombre_archivo = os.path.splitext(os.path.basename(ruta))[0]
    
    # Leer el archivo y guardarlo en el diccionario
    dataframes_csv[nombre_archivo] = pd.read_csv(ruta)
    
    print(nombre_archivo)

Archivos CSV cargados:
cuisine
hours
parking
payment_methods
ratings
restaurantes_slp
restaurants
usercuisine
userpayment
users
usuarios_slp


Continuamos por observar los primeros 5 registros de cada trabla, para comprobar que se han leido correctamente los archivos CSV

In [4]:
# Observamos los primeros registros de cada DataFrame CSV
for nombre, df in dataframes_csv.items():
    print(f"Archivo: {nombre}")
    display(df.head())  # O df.info() si quieres más detalle
    print("-" * 60)

Archivo: cuisine


,placeID,Rcuisine
0,135110,Spanish
1,135109,Italian
2,135107,Latin_American
3,135106,Mexican
4,135105,Fast_Food


------------------------------------------------------------
Archivo: hours


,placeID,hours,days
0,135111,00:00-23:30;,Mon;Tue;Wed;Thu;Fri;
1,135111,00:00-23:30;,Sat;
2,135111,00:00-23:30;,Sun;
3,135110,08:00-19:00;,Mon;Tue;Wed;Thu;Fri;
4,135110,00:00-00:00;,Sat;


------------------------------------------------------------
Archivo: parking


...

------------------------------------------------------------
Archivo: users


,userID,latitude,longitude,smoker,drink_level,dress_preference,ambience,transport,marital_status,hijos,birth_year,interest,personality,religion,activity,color,weight,budget,height
0,U1001,22.139997,-100.978803,false,abstemious,informal,family,on foot,single,independent,1989,variety,thrifty-protector,none,student,black,69,medium,1.77
1,U1002,22.150087,-100.983325,false,abstemious,informal,family,public,single,independent,1990,technology,hunter-ostentatious,Catholic,student,red,40,low,1.87
2,U1003,22.119847,-100.946527,false,social drinker,formal,family,public,single,independent,1989,none,hard-worker,Catholic,student,blue,60,low,1.69
3,U1004,18.867000,-99.183000,false,abstemious,informal,family,public,single,independent,1940,variety,hard-worker,none,professional,green,44,medium,1.53
4,U1005,22.183477,-100.959891,false,abstemious,no preference,family,public,single,independent,1992,none,thrifty-protector,Catholic,student,black,65,medium,1.69


------------------------------------------------------------
Archivo: usuarios_slp


,userID,latitude_usuario,longitude_usuario,smoker,drink_level,dress_preference,ambience,transport,marital_status,hijos,birth_year,interest,personality,religion,activity,color,weight,budget,height
0,U1077,22.156469,-100.985540,0,social drinker,elegant,family,public,married,kids,1987,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71
1,U1015,22.126760,-100.905209,1,social drinker,informal,family,public,single,independent,1989,technology,thrifty-protector,Catholic,student,black,87,medium,1.67
2,U1083,22.133920,-101.028373,0,abstemious,informal,Desconocido,Desconocido,Desconocido,Desconocido,1981,none,hard-worker,none,Desconocido,yellow,40,low,1.60
3,U1108,22.143524,-100.987562,0,abstemious,informal,solitary,public,single,independent,1983,technology,thrifty-protector,Catholic,student,blue,76,medium,1.81
4,U1055,22.143289,-100.987683,0,abstemious,no preference,family,car owner,married,dependent,1952,none,hard-worker,Catholic,professional,blue,70,medium,1.66


------------------------------------------------------------


Ahora, procedemos a revisar las dimensiones de cada archivo CSV.

In [5]:
# Observamos las dimensiones de cada DataFrame CSV
for nombre, df in dataframes_csv.items():
    print(f"Tamaño de Dataframe {nombre}: {df.shape}")
    print("-" * 60)

Tamaño de Dataframe cuisine: (916, 2)
------------------------------------------------------------
Tamaño de Dataframe hours: (2339, 3)
------------------------------------------------------------
Tamaño de Dataframe parking: (702, 2)
------------------------------------------------------------
Tamaño de Dataframe payment_methods: (1314, 2)
------------------------------------------------------------
Tamaño de Dataframe ratings: (1161, 5)
------------------------------------------------------------
Tamaño de Dataframe restaurantes_slp: (74, 19)
------------------------------------------------------------
Tamaño de Dataframe restaurants: (130, 21)
------------------------------------------------------------
Tamaño de Dataframe usercuisine: (330, 2)
------------------------------------------------------------
Tamaño de Dataframe userpayment: (177, 2)
------------------------------------------------------------
Tamaño de Dataframe users: (138, 19)
-----------------------------------------

## 1.2 Archivos Excel

Hacemos lo mismo para archivos Excel. Los cargamos a otro diccionario. 

In [6]:
# Cargamos todos los archivos Excel en un diccionario de DataFrames
dataframes_excel = {}

print("Archivos Excel cargados:")
for ruta in archivos_excel:
    # Extraer el nombre del archivo, sin extensión
    nombre_archivo = os.path.splitext(os.path.basename(ruta))[0]
    
    # Leer el archivo y guardarlo en el diccionario
    dataframes_excel[nombre_archivo] = pd.read_excel(ruta)
    
    print(nombre_archivo)

Archivos Excel cargados:


restaurants
users


Hacemos una pequeña visualización de los datos en estos archivos.

In [7]:
# Observamos los primeros registros de cada DataFrame Excel
for nombre, df in dataframes_excel.items():
    print(f"Archivo: {nombre}")
    display(df.head())  # O df.info() si quieres más detalle
    print("-" * 60)

Archivo: restaurants


,placeID,latitude,longitude,the_geom_meter,name,address,city,state,country,fax,...,alcohol,smoking_area,dress_code,accessibility,price,url,Rambience,franchise,area,other_services
0,134999,18.915421,-99.184871,0101000020957F000088568DE356715AC138C0A525FC46...,Kiku Cuernavaca,Revolucion,Cuernavaca,Morelos,Mexico,?,...,No_Alcohol_Served,none,informal,no_accessibility,medium,kikucuernavaca.com.mx,familiar,f,closed,none
1,132825,22.147392,-100.983092,0101000020957F00001AD016568C4858C1243261274BA5...,puesto de tacos,esquina santos degollado y leon guzman,s.l.p.,s.l.p.,mexico,?,...,No_Alcohol_Served,none,informal,completely,low,?,familiar,f,open,none
2,135106,22.149709,-100.976093,0101000020957F0000649D6F21634858C119AE9BF528A3...,El Rincón de San Francisco,Universidad 169,San Luis Potosi,San Luis Potosi,Mexico,?,...,Wine-Beer,only at bar,informal,partially,medium,?,familiar,f,open,none
3,132667,23.752697,-99.163359,0101000020957F00005D67BCDDED8157C1222A2DC8D84D...,little pizza Emilio Portes Gil,calle emilio portes gil,victoria,tamaulipas,?,?,...,No_Alcohol_Served,none,informal,completely,low,?,familiar,t,closed,none
4,132613,23.752903,-99.165076,0101000020957F00008EBA2D06DC8157C194E03B7B504E...,carnitas_mata,lic. Emilio portes gil,victoria,Tamaulipas,Mexico,?,...,No_Alcohol_Served,permitted,informal,completely,medium,?,familiar,t,closed,none


------------------------------------------------------------
Archivo: users


,userID,latitude,longitude,smoker,drink_level,dress_preference,ambience,transport,marital_status,hijos,birth_year,interest,personality,religion,activity,color,weight,budget,height
0,U1001,22.139997,-100.978803,False,abstemious,informal,family,on foot,single,independent,1989,variety,thrifty-protector,none,student,black,69,medium,1.77
1,U1002,22.150087,-100.983325,False,abstemious,informal,family,public,single,independent,1990,technology,hunter-ostentatious,Catholic,student,red,40,low,1.87
2,U1003,22.119847,-100.946527,False,social drinker,formal,family,public,single,independent,1989,none,hard-worker,Catholic,student,blue,60,low,1.69
3,U1004,18.867000,-99.183000,False,abstemious,informal,family,public,single,independent,1940,variety,hard-worker,none,professional,green,44,medium,1.53
4,U1005,22.183477,-100.959891,False,abstemious,no preference,family,public,single,independent,1992,none,thrifty-protector,Catholic,student,black,65,medium,1.69


------------------------------------------------------------


Revisamos también las dimensiones de estos archivos. 

In [8]:
# Observamos las dimensiones de cada DataFrame Excel
for nombre, df in dataframes_excel.items():
    print(f"Tamaño de Dataframe {nombre}: {df.shape}")
    print("-" * 60)

Tamaño de Dataframe restaurants: (130, 21)
------------------------------------------------------------
Tamaño de Dataframe users: (138, 19)
------------------------------------------------------------


## 1.3 Comparación de Arvhivos CSV y Excel 

### 1.3.1 Restaurantes

Verificamos que cada columna de los archivos restaurats.csv y restaurants.xlsx tenga el mismo número de valores únicos. 

In [9]:
dataframes_csv['restaurants'].nunique() != dataframes_excel['restaurants'].nunique()

placeID           False
latitude          False
longitude         False
the_geom_meter    False
name              False
address           False
city               True
state             False
country           False
fax               False
zip               False
alcohol           False
smoking_area      False
dress_code        False
accessibility     False
price             False
url               False
Rambience         False
franchise         False
area              False
other_services    False
dtype: bool

Notamos que difieren en la columna de ciudad. Ahora, checamos el número de valores únicos en esta columna en cada dataframe.

In [10]:
dataframes_csv['restaurants'].columns != dataframes_excel['restaurants'].columns

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False])

In [11]:
print("Número de valores únicos en columna de ciudad, para archivo CSV:",dataframes_csv['restaurants']['city'].nunique()) # Cuenta los valores únicos en la columna 'city'

Número de valores únicos en columna de ciudad, para archivo CSV: 17


In [12]:
print("Número de valores únicos en columna de ciudad, para archivo Excel:",dataframes_excel['restaurants']['city'].nunique()) # Cuenta los valores únicos en la columna 'city'

Número de valores únicos en columna de ciudad, para archivo Excel: 15


Definimos una función que nos ayudará a revisar los porcentajes de valores nulos en cada columna. 

In [13]:
# Checa porcentajes de valores nulos en columnas
def completitud_datos(df):
    return df.isnull().sum().sort_values(ascending=False) / df.shape[0]

In [14]:
completitud_datos(dataframes_csv['restaurants']) != completitud_datos(dataframes_excel['restaurants'])

placeID           False
latitude          False
longitude         False
the_geom_meter    False
name              False
address           False
city              False
state             False
country           False
fax               False
zip               False
alcohol           False
smoking_area      False
dress_code        False
accessibility     False
price             False
url               False
Rambience         False
franchise         False
area              False
other_services    False
dtype: bool

Notamos que ambas tablas tienen el mismo porcentaje de nulos. Sin embargo, como el archivo CSV contiene un poco más de información referente a ciudades (y en todo lo demás poseen la misma información), optamos por usar el archivo 'restaurants.csv' para nuestra ingeniería y análisis de datos. 

### 1.3.2 Usuarios

Ahora, nos toca revisar las tablas de 'users.csv' y 'users.xlsx'. Queremos ver si difieren en algún tipo de información. 

In [15]:
dataframes_csv['users'].nunique() != dataframes_excel['users'].nunique()

userID              False
latitude            False
longitude           False
smoker              False
drink_level         False
dress_preference    False
ambience            False
transport           False
marital_status      False
hijos               False
birth_year          False
interest            False
personality         False
religion            False
activity            False
color               False
weight              False
budget              False
height              False
dtype: bool

In [16]:
dataframes_csv['users'].columns != dataframes_excel['users'].columns

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [17]:
completitud_datos(dataframes_csv['users']) != completitud_datos(dataframes_excel['users'])

userID              False
latitude            False
longitude           False
smoker              False
drink_level         False
dress_preference    False
ambience            False
transport           False
marital_status      False
hijos               False
birth_year          False
interest            False
personality         False
religion            False
activity            False
color               False
weight              False
budget              False
height              False
dtype: bool

Dado que ambas tablas de usuarios no difieren en información, optamos por ignorar estos archivos Excel y continuamos todo el tratamiento de datos con los archivos CSV. 

### 1.3.3 Renombramos Columnas

Para hacer toda la ingeniería y limpieza de datos, crearemos una tabla analítica que contenga toda la información referente a San Luis Potosí. Antes de hacer eso, cambiamos los nombres de ciertas columnas que tienen el mismo nombre a otras columnas en otras tablas. 

In [18]:
#Cambiamos nombre para columnas de latitud y longitud de restaurantes 
dataframes_csv['restaurants'] = dataframes_csv['restaurants'].rename(columns={
    "latitude": "latitude_rest",
    "longitude": "longitude_rest"
    })

In [19]:
# #Cambiamos nombre para columnas de cocina ofrecida en Restaurante 
dataframes_csv['cuisine'] = dataframes_csv['cuisine'].rename(columns={"Rcuisine": "Rcuisine_rest",})

In [20]:
#Cambiamos nombre para columnas de latitud y longitud de usuarios 
dataframes_csv['users'] = dataframes_csv['users'].rename(columns={
    "latitude": "latitude_usuario",
    "longitude": "longitude_usuario",
    })

In [21]:
#Cambiamos nombre para columnas de cocina preferifa del usuario 
dataframes_csv['usercuisine'] = dataframes_csv['usercuisine'].rename(columns={"Rcuisine": "Rcuisine_user",})

In [22]:
# Hacemos una copia de las tablas previo a la extracción de unidad muestral
df_restaurantes = dataframes_csv['restaurants'].copy()
df_users = dataframes_csv['users'].copy()

## 1.4 Filtrado de Datos por Unidad Muestral

Dado que, planeamos restringir nuestros datos a la unidad muestral de restaurantes en San Luis Potosí, verificamos que la columna de *ciudad* en la tabla de **restaurantes**, contenga un formato unificado y claro. Además, también necesitamos ver qué otras ciudades están contempladas en este conjunto de datos, pues más adelante debemos verificar que el conjunto de datos restringido a nuestra unidad muestral represente más del 10% de la información total. 

In [23]:
# Muestra los valores que toma la variable de ciudad en la tabla restaurantes
dataframes_csv['restaurants']['city'].unique() 

array(['Cuernavaca', 's.l.p.', 'San Luis Potosi', 'victoria ', 'victoria',
       'Cd Victoria', '?', 'san luis potosi', 'Jiutepec', 'cuernavaca',
       'slp', 'Soledad', 'san luis potos', 'san luis potosi ',
       'Ciudad Victoria', 'Cd. Victoria', 's.l.p'], dtype=object)

Podemos notar que no hay un formato claro en esta columna. Varias cuidades aparecen en varios registros con nombres distintos. Esto dificultara la extracción de datos restringida a la unidad muestral seleccionada. Por ello, procedemos a limpiar esta columna con valores unificados por ciudad. 

Para lograr esto, nos apoyaremos de un diccionario, en el cuál, por cada valor único de esta columna, se clasificará la cuidad que representa este valor. 

In [24]:
# Creamos diccionario para unificar nombres de ciudades
mapa_ciudades = {
    'slp': 'San Luis Potosí',
    's.l.p.': 'San Luis Potosí',
    's.l.p': 'San Luis Potosí',
    'san luis potosi': 'San Luis Potosí',
    'san luis potos': 'San Luis Potosí',
    'san luis potosi ': 'San Luis Potosí',
    'san luis potosi': 'San Luis Potosí',

    'victoria': 'Ciudad Victoria',
    'victoria ': 'Ciudad Victoria',
    'cd victoria': 'Ciudad Victoria',
    'cd. victoria': 'Ciudad Victoria',
    'ciudad victoria': 'Ciudad Victoria',

    'cuernavaca': 'Cuernavaca',
    'jiutepec': 'Jiutepec',
    'soledad': 'Soledad',
    '?': None
}


Para facilitar el mapeo de ciudades según su nombre correcto, debemos estandarizar los nombres de las cuidades. Nos apoyamos de métodos de la clase `String` como `strip()` y `lower()` para elimirar espacios y pasar todo a minúsculas. También haremos uso del método `replace()`, el cual nos ayudará a eliminar los puntos de cada cadena de texto. 

In [25]:
# Función para estandarizar los nombres de las ciudades
def estandarizar_ciudad(ciudad):
    if pd.isna(ciudad):
        return None # Si es NaN, devolvemos None
    ciudad_limpia = ciudad.strip().lower().replace('.', '')  # Convertimos a minúsculas y eliminamos espacios al inicio y al final
    return mapa_ciudades.get(ciudad_limpia, ciudad.strip())  # Si no está en el mapa, deja el original

In [26]:
# Aplicamos la función de estandarización a la columna 'city' del DataFrame 'restaurants'
dataframes_csv['restaurants']['city'] = dataframes_csv['restaurants']['city'].apply(estandarizar_ciudad)

In [27]:
# Comprobamos los valores únicos de la columna 'city' después de la estandarización
print(dataframes_csv['restaurants']['city'].unique())

['Cuernavaca' 'San Luis Potosí' 'Ciudad Victoria' None 'Jiutepec'
 'Soledad']


Podemos observar que ahora, los valores de ciudades están ordenados. Por lo tanto, ahora podemos continuar con extraer la unidad muestral que escogimos. 

In [28]:
# Definimos una variable para el DataFrame de restaurantes
restaurants = dataframes_csv['restaurants']

# Filtramos los restaurantes de San Luis Potosí (suponiendo que ya limpiaste la ciudad en una columna)
placeIDs_slp = restaurants[restaurants['city'] == 'San Luis Potosí']['placeID'].unique()
print(f"Número de restaurantes en San Luis Potosí: {len(placeIDs_slp)}")

Número de restaurantes en San Luis Potosí: 74


In [29]:
# Nombres de tablas que usan placeID
tablas_place = ['cuisine', 'hours', 'parking', 'payment_methods', 'ratings','restaurants']

# Diccionario para almacenar las tablas filtradas
filtradas_por_lugar = {}

for nombre in tablas_place:
    df = dataframes_csv[nombre]
    filtradas_por_lugar[nombre] = df[df['placeID'].isin(placeIDs_slp)]

In [30]:
userIDs_slp = filtradas_por_lugar['ratings']['userID'].unique()

In [31]:
tablas_user = ['users', 'usercuisine', 'userpayment']
filtradas_por_user = {}

for nombre in tablas_user:
    df = dataframes_csv[nombre]
    filtradas_por_user[nombre] = df[df['userID'].isin(userIDs_slp)]

In [32]:
resumen_porcentajes = {}

# Por tablas filtradas por placeID
for nombre in tablas_place:
    total = dataframes_csv[nombre].shape[0]
    subset = filtradas_por_lugar[nombre].shape[0]
    resumen_porcentajes[nombre] = round(100 * subset / total, 2)

# Por tablas filtradas por userID
for nombre in tablas_user:
    total = dataframes_csv[nombre].shape[0]
    subset = filtradas_por_user[nombre].shape[0]
    resumen_porcentajes[nombre] = round(100 * subset / total, 2)

# Tabla principal de restaurantes
total_restaurantes = dataframes_csv['restaurants'].shape[0]
subset_restaurantes = len(placeIDs_slp)
resumen_porcentajes['restaurants'] = round(100 * subset_restaurantes / total_restaurantes, 2)

In [33]:
from pprint import pprint

print("Porcentaje de registros correspondientes a San Luis Potosí por tabla:")
pprint(resumen_porcentajes)

Porcentaje de registros correspondientes a San Luis Potosí por tabla:
{'cuisine': 7.42,
 'hours': 9.49,
 'parking': 10.54,
 'payment_methods': 13.24,
 'ratings': 71.83,
 'restaurants': 56.92,
 'usercuisine': 80.91,
 'userpayment': 61.02,
 'users': 65.22}


Podemos notar que, en efecto, la información referente a la unidad muestral elegida representa más del 10% de la información total. Esto nos permite seguir con la extracción de información referente a la unidad muestral.  

In [34]:
# Comprobamos que solo tenemos ahora datos de SLP
filtradas_por_lugar['restaurants']['city'].unique() 

array(['San Luis Potosí'], dtype=object)

In [35]:
ratings_slp = filtradas_por_lugar['ratings'].copy()

In [36]:
# Asegurarnos que 'placeID' sea string en todos los dataframes que lo contienen
ratings_slp['placeID'] = ratings_slp['placeID'].astype(str)
dataframes_csv['restaurants']['placeID'] = dataframes_csv['restaurants']['placeID'].astype(str)
dataframes_csv['parking']['placeID'] = dataframes_csv['parking']['placeID'].astype(str)
dataframes_csv['cuisine']['placeID'] = dataframes_csv['cuisine']['placeID'].astype(str)
dataframes_csv['payment_methods']['placeID'] = dataframes_csv['payment_methods']['placeID'].astype(str)
dataframes_csv['hours']['placeID'] = dataframes_csv['hours']['placeID'].astype(str)

In [37]:
# Ahora sí puedes hacer el merge
merged = ratings_slp.merge(dataframes_csv['restaurants'], on='placeID', how='left')

# Merge con el resto
for nombre in ['parking', 'cuisine', 'payment_methods', 'hours']:
    dataframes_csv[nombre]['placeID'] = dataframes_csv[nombre]['placeID'].astype(str)
    merged = merged.merge(dataframes_csv[nombre], on='placeID', how='left')

In [38]:
print("Registros originales:", len(ratings_slp))
print("Registros después del merge:", len(merged))

Registros originales: 834
Registros después del merge: 6591


In [39]:
# Merge con users
merged = merged.merge(dataframes_csv['users'], on='userID', how='left')

# Merge con usercuisine y userpayment
for nombre in ['usercuisine', 'userpayment']:
    dataframes_csv[nombre]['userID'] = dataframes_csv[nombre]['userID'].astype(str)
    merged = merged.merge(dataframes_csv[nombre], on='userID', how='left')

In [40]:
merged.head()

,userID,placeID,rating,food_rating,service_rating,latitude_rest,longitude_rest,the_geom_meter,name,address,...,interest,personality,religion,activity,color,weight,budget,height,Rcuisine_user,Upayment
0,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71,Mexican,VISA
1,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71,Mexican,cash
2,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71,Mexican,bank_debit_cards
3,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71,Mexican,VISA
4,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,technology,thrifty-protector,Catholic,student,blue,65,medium,1.71,Mexican,cash


In [41]:
# Revisamos los registros disponibles para la cuidad de SLP
merged.shape

(28434, 50)

In [42]:
print("Número de Restaurantes Disponibles para análisis:",merged['name'].nunique())

Número de Restaurantes Disponibles para análisis: 74


Porcedemos a revisar si existen valores nulos en nuestra tabla. 

In [43]:
# Revisamos la completitud de los datos
print("Numero de Columnas con valores Nulos: ",sum(completitud_datos(merged) != 0))

Numero de Columnas con valores Nulos:  3


In [44]:
print("Porcentaje de valores nulos por columna: ")
print(completitud_datos(merged))

Porcentaje de valores nulos por columna: 
Rcuisine_rest        0.132728
Upayment             0.013927
Rpayment             0.004748
rating               0.000000
service_rating       0.000000
latitude_rest        0.000000
longitude_rest       0.000000
food_rating          0.000000
name                 0.000000
address              0.000000
city                 0.000000
state                0.000000
country              0.000000
fax                  0.000000
zip                  0.000000
the_geom_meter       0.000000
userID               0.000000
placeID              0.000000
dress_code           0.000000
smoking_area         0.000000
alcohol              0.000000
accessibility        0.000000
franchise            0.000000
area                 0.000000
url                  0.000000
price                0.000000
parking_lot          0.000000
other_services       0.000000
hours                0.000000
days                 0.000000
latitude_usuario     0.000000
longitude_usuario    0.00000

Notamos que, en efecto, debemos tratar algunos valores nulos de tres columnas. Estas tres columnas representan datos categóricos. Para algunas columnas, imputaremos con la moda, pero para otras columnas, agregaremos otra clase llamada "Desconocido" pues en estos casos, la inforamción perdida también nos aporta cierta información de nuestros datos. 

In [45]:
# Imputamos valores faltantes de Rcuisine_x como una categoría "Desconocida"
merged['Rcuisine_rest'] = merged['Rcuisine_rest'].fillna('Desconocido')

In [46]:
# Importamos SimpleImputer de sklearn 
from sklearn.impute import SimpleImputer

# Instancuamos los imputadores con la estrategia de la media y moda
imputador_media = SimpleImputer(strategy='mean')
imputador_moda = SimpleImputer(strategy='most_frequent')

# Imputamos los valores faltantes en la columna 'Upayment'
merged[['Upayment']] = imputador_moda.fit_transform(merged[['Upayment']])

# Imputamos los valores faltantes en la columna 'Rpayment'
merged[['Rpayment']] = imputador_moda.fit_transform(merged[['Rpayment']])

In [47]:
completitud_datos(merged)

userID               0.0
placeID              0.0
rating               0.0
food_rating          0.0
service_rating       0.0
latitude_rest        0.0
longitude_rest       0.0
the_geom_meter       0.0
name                 0.0
address              0.0
city                 0.0
state                0.0
country              0.0
fax                  0.0
zip                  0.0
alcohol              0.0
smoking_area         0.0
dress_code           0.0
accessibility        0.0
price                0.0
url                  0.0
Rambience            0.0
franchise            0.0
area                 0.0
other_services       0.0
parking_lot          0.0
Rcuisine_rest        0.0
Rpayment             0.0
hours                0.0
days                 0.0
latitude_usuario     0.0
longitude_usuario    0.0
smoker               0.0
drink_level          0.0
dress_preference     0.0
ambience             0.0
transport            0.0
marital_status       0.0
hijos                0.0
birth_year           0.0


Podemos notar que, hemos removido los valores nulos de estas tres columnas correctamente. Sin embargo, existen otras columnas como "budget" que contienen algo similar a un valor nulo. 

In [48]:
merged['budget'].unique()

array(['medium', '?', 'low', 'high'], dtype=object)

In [49]:
merged['budget'].value_counts() # Asignamos el valor más frecuente a los NaN

budget
low       14664
medium    12789
high        492
?           489
Name: count, dtype: int64

Este tipo de columnas contienen valores "?", los cuales deben ser tratados de la misma manera que un valor nulo. Pero antes de ello, debemos convertir estos valores de interrogación en un valores nulos, y con ello, podremos imputarlos coreectamente. 

In [50]:
# Función que da información de variables categoricas 
def revisar_categoricas(df):
    columnas_categoricas = df.select_dtypes(include=['object']).columns
    for col in columnas_categoricas:
        print(f"Columna: {col}")
        print(df[col].value_counts(dropna=False))
        print("-" * 40)

Podemos ver que, en efecto, varias columas categóricas contienen en valor de interrogación. 

In [51]:
revisar_categoricas(merged)

Columna: userID
userID
U1135    10815
U1108     4536
U1004     1566
U1016     1428
U1101     1395
         ...  
U1029       42
U1064       39
U1131       30
U1102       21
U1052       18
Name: count, Length: 90, dtype: int64
----------------------------------------
Columna: placeID
placeID
135032    4194
135052    3060
135028    2256
135058    1647
135045    1584
          ... 
132845      18
132937      18
135049      15
132870      15
135044      12
Name: count, Length: 74, dtype: int64
----------------------------------------
Columna: the_geom_meter
the_geom_meter
0101000020957F000017D69FF5084858C1A8F2188740A24B41    4194
0101000020957F0000B72147F1274858C189A406ED75A34B41    3060
0101000020957F0000B4F00BDD8B4858C10AC3619C83A64B41    2256
0101000020957F0000EB71019D424558C15CC5404365A94B41    1647
0101000020957F00000B6735CA004858C108FD525CB2A44B41    1584
                                                      ... 
0101000020957F0000B230F8670C4E58C1D590A2700F8F4B41      18
010100002095

Procedemos a definir una función que nos permita sustituir estos valores de interrogación por valores NAN de la librería Numpy. 

In [52]:
import numpy as np

# Función para reemplazar interrogaciones por NaN en columnas de tipo objeto o categoría
def reemplazar_interrogacion(df):
    df_cleaned = df.copy()

    for col in df_cleaned.select_dtypes(include=['object', 'category']).columns:
        df_cleaned[col] = df_cleaned[col].replace('?', np.nan)

    return df_cleaned

In [53]:
df = reemplazar_interrogacion(merged)

C:\Users\Aura De La Garza G\AppData\Local\Temp\ipykernel_8448\2620907506.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned[col] = df_cleaned[col].replace('?', np.nan)


Una vez que sustituimos los valores de interrogacion por valores NAN, procedemos a revisar la completitud de nuestros datos de nuevo. 

In [54]:
completitud_datos(df)

fax                  1.000000
url                  0.775164
zip                  0.249103
country              0.031019
hijos                0.020679
budget               0.017198
address              0.014666
dress_preference     0.013716
ambience             0.013610
activity             0.011711
marital_status       0.009918
transport            0.009918
smoker               0.007702
state                0.002216
placeID              0.000000
rating               0.000000
service_rating       0.000000
userID               0.000000
dress_code           0.000000
smoking_area         0.000000
alcohol              0.000000
city                 0.000000
latitude_rest        0.000000
longitude_rest       0.000000
the_geom_meter       0.000000
name                 0.000000
food_rating          0.000000
parking_lot          0.000000
price                0.000000
accessibility        0.000000
franchise            0.000000
area                 0.000000
other_services       0.000000
Rambience 

Podemos ver que, debemos imputar algunos otros valores nulos. De nuevo, algunos casos se imputaran con moda o con clase "Desconocida", excepto las columnas de "state" y "country", pues para estas columnas, solo puede haber un tipo de valor. 

In [55]:
df['state'].unique()

array(['SLP', 's.l.p.', 'San Luis Potosi', 'S.L.P.', nan,
       'san luis potosi', 'slp', 'mexico', 'san luis potos'], dtype=object)

In [56]:
df['country'].unique()

array(['Mexico', 'mexico', nan], dtype=object)

In [57]:
df['state'] = merged['state'].apply(lambda x: 'San Luis Potosí')
df['country'] = merged['country'].apply(lambda x: 'México')

In [58]:
df['state'].unique()  

array(['San Luis Potosí'], dtype=object)

In [59]:
df['country'].unique()

array(['México'], dtype=object)

Revisamos nuevamente las columnas con valores nulos. De acuerdo a nuestra conveniencia, seleccionaremos el método para imputar los valores nulos. 

In [60]:
completitud_datos(df)

fax                  1.000000
url                  0.775164
zip                  0.249103
hijos                0.020679
budget               0.017198
address              0.014666
dress_preference     0.013716
ambience             0.013610
activity             0.011711
transport            0.009918
marital_status       0.009918
smoker               0.007702
placeID              0.000000
userID               0.000000
latitude_rest        0.000000
longitude_rest       0.000000
service_rating       0.000000
rating               0.000000
dress_code           0.000000
smoking_area         0.000000
alcohol              0.000000
country              0.000000
city                 0.000000
state                0.000000
the_geom_meter       0.000000
name                 0.000000
food_rating          0.000000
parking_lot          0.000000
price                0.000000
accessibility        0.000000
franchise            0.000000
area                 0.000000
other_services       0.000000
Rambience 

Recordemos que, las columnas que contengan más del 65% de nulos, necesitaremos eliminarlas. Para las demás columnas, imputamos con moda y clase "Desconocida", pues, de nuevo todas las variables que hace falta imputar son categóricas.

In [61]:
# 1. Eliminar columnas con más del 65% de valores nulos
df.drop(columns=['fax', 'url'], inplace=True)

# 2. Imputar con moda
df[['zip']] = imputador_moda.fit_transform(df[['zip']])
df[['budget']] = imputador_moda.fit_transform(df[['budget']])
df[['dress_preference']] = imputador_moda.fit_transform(df[['dress_preference']])
df[['smoker']] = imputador_moda.fit_transform(df[['smoker']])

# 3. Imputar con "Desconocido"
columnas_desconocido = [
    'hijos', 'address', 'ambience', 'activity', 
    'transport', 'marital_status'
]

for col in columnas_desconocido:
    df[col].fillna("Desconocido", inplace=True)

C:\Users\Aura De La Garza G\AppData\Local\Temp\ipykernel_8448\1367496328.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("Desconocido", inplace=True)


In [62]:
completitud_datos(df)

userID               0.0
placeID              0.0
rating               0.0
food_rating          0.0
service_rating       0.0
latitude_rest        0.0
longitude_rest       0.0
the_geom_meter       0.0
name                 0.0
address              0.0
city                 0.0
state                0.0
country              0.0
zip                  0.0
alcohol              0.0
smoking_area         0.0
dress_code           0.0
accessibility        0.0
price                0.0
Rambience            0.0
franchise            0.0
area                 0.0
other_services       0.0
parking_lot          0.0
Rcuisine_rest        0.0
Rpayment             0.0
hours                0.0
days                 0.0
latitude_usuario     0.0
longitude_usuario    0.0
smoker               0.0
drink_level          0.0
dress_preference     0.0
ambience             0.0
transport            0.0
marital_status       0.0
hijos                0.0
birth_year           0.0
interest             0.0
personality          0.0


Como podemos ver, hemos eliminado exitosamente valores faltantes de nuestra tabla. Ahora, procedemos a revisar algunas columnas que nos convendrá convertir a valores numéricos. Comenzamos por checar las variables binarias en nuestra tabla. 

### 1.4.1 Variables Binarias

Creamos una función que nos ayudara a revisar si existen variables binarias y ver que tipo de datos contienen. Además podemos revisar los valores únicos. 

In [63]:
def checar_var_binarias(df):
    """
    Esta función imprime información sobre las columnas categóricas que tienen exactamente dos categorías.
    """
    for col in df.columns:
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            if df[col].nunique() == 2:
                print(f"Columna {col} tiene 2 categorías, convirtiendo a variable binaria.")
                print(f"Tipo de variable: {df[col].dtype}")
                print(f"Valores únicos: {df[col].unique()}") 
            else:
                continue
        else:
            if df[col].nunique() == 2:
                print(f"Columna {col} tiene 2 categorías, convirtiendo a variable binaria.")
                print(f"Tipo de variable: {df[col].dtype}")
                print(f"Valores únicos: {df[col].unique()}")

In [64]:
checar_var_binarias(df)

Columna dress_code tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: object
Valores únicos: ['informal' 'casual']
Columna Rambience tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: object
Valores únicos: ['familiar' 'quiet']
Columna franchise tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: object
Valores únicos: ['f' 't']
Columna area tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: object
Valores únicos: ['closed' 'open']
Columna smoker tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: object
Valores únicos: ['false' 'true']


Para mayor simplicidad, transformaremos los valores de estas variables binarias en ceros y unos (que representan No y Sí, respectivamente). Estas variables serán ahora variables aleatorias Bernoulli, con algún parámetro $p\in(0,1)$ que se peude obtener de la frecuencia de cada clase. 

In [65]:
def variables_binarias(df):
    """
    Esta función convierte las columnas categóricas con dos categorías en variables binarias.
    """
    columnas_binarias = df.select_dtypes(include=['object']).columns
    for col in columnas_binarias:
        if df[col].nunique() == 2:
            df[col] = df[col].map({df[col].unique()[0]: 0, df[col].unique()[1]: 1})
    return df

In [66]:
df = variables_binarias(df)

Podemos comprobar que, ahora, nuestras variables binarias, son variables aleatorias Bernoulli. 

In [67]:
checar_var_binarias(df)

Columna dress_code tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: int64
Valores únicos: [0 1]
Columna Rambience tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: int64
Valores únicos: [0 1]
Columna franchise tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: int64
Valores únicos: [0 1]
Columna area tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: int64
Valores únicos: [0 1]
Columna smoker tiene 2 categorías, convirtiendo a variable binaria.
Tipo de variable: int64
Valores únicos: [0 1]


### 1.4.2 Variables Discretas

Parecido a lo que hicimos con las variables binarias, ahora creamos una función que nos permita revisar cierta información de las variables candidatas a ser variables aleatorias discretas. 

In [68]:
def checar_var_discretas(df, max_categorias):
    """
    Esta función imprime información sobre las columnas categóricas que tienen exactamente dos categorías.
    """
    
    for col in df.columns:
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            if  df[col].nunique() <= max_categorias:
                if df[col].nunique() >= 3:
                    print(f"Columna {col} tiene de 3 a 16 categorías, convirtiendola en variable discreta.")
                    print(f"Tipo de variable: {df[col].dtype}")
                    print(f"Valores únicos: {df[col].unique()}") 
            else:
                continue
        else:
            if df[col].nunique() <= max_categorias:
                if df[col].nunique() >= 3:
                    print(f"Columna {col} tiene de 3 a 16 categorías, convirtiendola en variable discreta.")
                    print(f"Tipo de variable: {df[col].dtype}")
                    print(f"Valores únicos: {df[col].unique()}")

En este caso, decidimos que, las variables discretas son aquellas con a lo más 20 valores únicos posibles, aunque este número siempre se escogerá según la conveniencia para manejar bien nuestro datos. 

In [69]:
checar_var_discretas(df, 20)

Columna rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 1 0]
Columna food_rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 0 1]
Columna service_rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 1 0]
Columna alcohol tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['No_Alcohol_Served' 'Full_Bar' 'Wine-Beer']
Columna smoking_area tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['not permitted' 'section' 'none' 'only at bar' 'permitted']
Columna accessibility tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['no_accessibility' 'completely' 'partially']
Columna price tiene de 3 a 16 categorías, convirtiendola en variable discr

Notamos que existen ciertas variables discreatas que requieren ser convertidas a numeros enteros, pues esto nos peude favorecer para escoger mejores variables predictoras. 

In [70]:
# Mapeo ordinal para 'budget' y 'price'
ordinal_map = {'low': 1, 'medium': 2, 'high': 3}
df['budget_numerico'] = df['budget'].map(ordinal_map)
df['price_numerico'] = df['price'].map(ordinal_map)

# Mapeo para 'drink_level'
drink_map = {'abstemious': 1, 'casual drinker': 2, 'social drinker': 3}
df['drink_level_numerico'] = df['drink_level'].map(drink_map)

# Opcional: dress_preference
dress_map = {
    'no preference': 1,
    'informal': 2,
    'formal': 3,
    'elegant': 4
}
df['dress_preference_numerico'] = df['dress_preference'].map(dress_map)


In [71]:
checar_var_discretas(df,20)

Columna rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 1 0]
Columna food_rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 0 1]
Columna service_rating tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: int64
Valores únicos: [2 1 0]
Columna alcohol tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['No_Alcohol_Served' 'Full_Bar' 'Wine-Beer']
Columna smoking_area tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['not permitted' 'section' 'none' 'only at bar' 'permitted']
Columna accessibility tiene de 3 a 16 categorías, convirtiendola en variable discreta.
Tipo de variable: object
Valores únicos: ['no_accessibility' 'completely' 'partially']
Columna price tiene de 3 a 16 categorías, convirtiendola en variable discr

In [72]:
df_pre_variables = df.copy()

# 2. Creación de Nuevas Variables 

Ahora que nuestras variables estan limpias de valores faltantes y que el formato de algunas columnas se ajusto a conveniencia, procedemos a crear las nuevas variables que nos permitiran medir el exito de los restaurantes en San Luis Potosí. 

### 2.1 Variables Adicionales

Definimos variables adicionales que nos puedan ayudar a entender el desempeño de los restaurantes en San Luis Potosí. 

In [73]:
# 1. Tiene estacionamiento
df['tiene_estacionamiento'] = df['parking_lot'].apply(lambda x: 0 if str(x).lower() == 'none' else 1)

# 2. Diversidad de cocina (cuántos tipos de cocina diferentes ofrece el restaurante)
diversidad = df.groupby('placeID')['Rcuisine_rest'].nunique().rename('diversidad_cocina')
df = df.merge(diversidad, on='placeID', how='left')


# 3. Afinidad usuario-restaurante en tipos de cocina
def afinidad_usuario_cocina(preferidas, ofrecidas):
    if pd.isna(preferidas) or pd.isna(ofrecidas):
        return 0
    preferidas_set = set(preferidas.lower().split(','))
    ofrecidas_set = set(ofrecidas.lower().split(','))
    return int(len(preferidas_set.intersection(ofrecidas_set)) > 0)

df['afinidad_usuario_cocina'] = df.apply(
    lambda row: afinidad_usuario_cocina(row['Rcuisine_user'], row['Rcuisine_rest']),
    axis=1
)

# 4. Presupuesto compatible: budget usuario >= price del restaurante
df['presupuesto_compatible'] = (df['budget_numerico'] >= df['price_numerico']).astype(int)



In [74]:
from datetime import datetime

# 5. Frecuencia de visitas por usuario
visitas = df.groupby('userID').size().rename('frecuencia_visitas_usuario')
df = df.merge(visitas, on='userID', how='left')

# 6. Calcular edad a partir del año de nacimiento
anio_actual = datetime.now().year
df['edad'] = anio_actual - df['birth_year']

# 7. Grupo etario
def clasificar_grupo_edad(edad):
    if edad < 25:
        return 'joven'
    elif edad <= 45:
        return 'adulto'
    else:
        return 'mayor'

df['grupo_edad'] = df['edad'].apply(clasificar_grupo_edad)

# 8. Preferencia de cocina por usuario (modo)
preferencia = df.groupby('userID')['Rcuisine_user'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
preferencia = preferencia.rename('preferencia_cocina')
df = df.merge(preferencia, on='userID', how='left')

# 9. Clientes únicos por restaurante
clientes = df.groupby('placeID')['userID'].nunique().rename('clientes_totales_restaurante')
df = df.merge(clientes, on='placeID', how='left')

In [75]:
# 10: numero de personas por codigo postal
# Creamos una dataframe que contenga solo los usuarios con las nuevas columnas 
df_users_loc = df[['latitude_usuario', 'longitude_usuario', 'zip']].dropna()

# Agrupamos por codigo postal y contamos los usuarios en estos
users_per_zipcode = df_users_loc['zip'].value_counts().reset_index()
users_per_zipcode.columns = ['zip', 'user_count']

# 11: Numero de restaurantes por codigo postal ---

# Creamos una dataframe que solo contega info de ubicación de restaurantes
df_restaurants_loc = df[['latitude_rest', 'longitude_rest', 'zip', 'placeID']].dropna()

# Agrupamos por codigo postal y contamos el número de restaurantes. 
restaurants_per_zipcode = df_restaurants_loc.groupby('zip')['placeID'].nunique().reset_index()
restaurants_per_zipcode.columns = ['zip', 'restaurant_count']

# Juntamos las nuevas variables en la dataframe original 

# Introducimos la cuenta de usuarios por codigo postal 
df = df.merge(users_per_zipcode, on='zip', how='left')

# Introducimos la cuenta de restaurantes por codigo postal 
df = df.merge(restaurants_per_zipcode, on='zip', how='left')

# Observamos el dataframe actualizado 
print("Dataframe actualizado:")
df.head()


Dataframe actualizado:


,userID,placeID,rating,food_rating,service_rating,latitude_rest,longitude_rest,the_geom_meter,name,address,...,diversidad_cocina,afinidad_usuario_cocina,presupuesto_compatible,frecuencia_visitas_usuario,edad,grupo_edad,preferencia_cocina,clientes_totales_restaurante,user_count,restaurant_count
0,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,0,1,72,38,adulto,Mexican,36,19986,40
1,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,0,1,72,38,adulto,Mexican,36,19986,40
2,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,0,1,72,38,adulto,Mexican,36,19986,40
3,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,0,1,72,38,adulto,Mexican,36,19986,40
4,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,0,1,72,38,adulto,Mexican,36,19986,40


### 2.2 Variable Objetivo

Ahora que tenemos variables adicionales, podemos definir nuestra variable objetivo. Esta variable objetivo nos dira el nivel de éxito en general que tienen los restaurantes según sus ratings y su nivel de visitas. Esta variable se calcula de la siguiente manera: 

•	15% 'service_rating',

•	15% 'food_rating'

•	30% 'rating' 

•	40% 'clientes_totales_restaurante'

Posterior a sumar estos valores, los normalizamos para que representen un porcentaje de éxito (valores en el intervalo $(0,1)$). Después de normalizarlos, los tranformamos en una variable de 3 clases (éxito bajo, éxito medio y éxito alto). Esta variable objetivo se llamara 'success_clase'. 

In [76]:
# 1. Crear variable weighted success
df['overall_success_raw'] = (
    0.15 * df['service_rating'] +
    0.15 * df['food_rating'] +
    0.30 * df['rating'] +
    0.40 * df['clientes_totales_restaurante']
)

# 2. Normalizar entre 0 y 1
min_val = df['overall_success_raw'].min()
max_val = df['overall_success_raw'].max()

df['overall_success'] = (
    df['overall_success_raw'] - min_val
) / (max_val - min_val)

# 3. Eliminar la columna auxiliar si ya no se necesita
df.drop(columns='overall_success_raw', inplace=True)

In [77]:
# Discretización en 3 clases
df['success_clase'] = pd.qcut(
    df['overall_success'], 
    q=3, 
    labels=['bajo', 'medio', 'alto']
)

In [78]:
# Elements in list_a but not in list_b
nuevas_variables = list(set(df.columns) - set(df_pre_variables.columns))
print(f"Nuevas Variables: {nuevas_variables}")
print("Número de nuevas variables:", len(nuevas_variables))

Nuevas Variables: ['tiene_estacionamiento', 'afinidad_usuario_cocina', 'presupuesto_compatible', 'preferencia_cocina', 'user_count', 'edad', 'frecuencia_visitas_usuario', 'grupo_edad', 'clientes_totales_restaurante', 'overall_success', 'diversidad_cocina', 'restaurant_count', 'success_clase']
Número de nuevas variables: 13


Visializamos el dataframe ya actualizado para ver las nuevas columnas (incluyendo la variable objetivo). 

In [79]:
df.head()

,userID,placeID,rating,food_rating,service_rating,latitude_rest,longitude_rest,the_geom_meter,name,address,...,presupuesto_compatible,frecuencia_visitas_usuario,edad,grupo_edad,preferencia_cocina,clientes_totales_restaurante,user_count,restaurant_count,overall_success,success_clase
0,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,72,38,adulto,Mexican,36,19986,40,1.0,alto
1,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,72,38,adulto,Mexican,36,19986,40,1.0,alto
2,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,72,38,adulto,Mexican,36,19986,40,1.0,alto
3,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,72,38,adulto,Mexican,36,19986,40,1.0,alto
4,U1077,135085,2,2,2,22.150802,-100.98268,0101000020957F00009F823DA6094858C18A2D4D37F9A4...,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,...,1,72,38,adulto,Mexican,36,19986,40,1.0,alto


Ahora, procedemos a revisar la información de las nuevas variables que acabamos de crear. 

In [80]:
nuevas_variables = ['tiene_estacionamiento', 'diversidad_cocina', 'clientes_totales_restaurante',
                    'afinidad_usuario_cocina', 'edad', 'grupo_edad', 'frecuencia_visitas_usuario',
                    'preferencia_cocina', 'presupuesto_compatible']

for col in nuevas_variables:
    print(f"Nueva variable: {col}")
    print(f"Tipo de variable: {df[col].dtype}")
    print(f"Valores únicos: {df[col].unique()}") 

Nueva variable: tiene_estacionamiento
Tipo de variable: int64
Valores únicos: [1 0]
Nueva variable: diversidad_cocina
Tipo de variable: int64
Valores únicos: [1 2 3]
Nueva variable: clientes_totales_restaurante
Tipo de variable: int64
Valores únicos: [36 24 32 22  9  5 11 13 15  4 28 10 18 14  8  6 20 21 12 25 17  7]
Nueva variable: afinidad_usuario_cocina
Tipo de variable: int64
Valores únicos: [0 1]
Nueva variable: edad
Tipo de variable: int64
Valores únicos: [38 36 44 42 73 35 34 95 33 85 39 43 40 37]
Nueva variable: grupo_edad
Tipo de variable: object
Valores únicos: ['adulto' 'mayor']
Nueva variable: frecuencia_visitas_usuario
Tipo de variable: int64
Valores únicos: [   72    54    63  4536    69   117    39    90    57   108    93   105
   276    84   135   225    66    78    81   114    87    60  1566    18
    45   102    99    48   126   192   483  1428   153   450   336   174
  1395 10815    51    21    75   132    42   282   150   147    30    96]
Nueva variable: preferencia

Ahora, para facilitar el trabajo, reordenaremos la tabla para que las columnas referentes a usuario queden al principio, seguidas por las variables referentes a restaurantes. Al último, irán las columnas de interacción (restaurantes con usuarios). 

In [81]:
user_cols = [
    'userID', 'latitude_usuario', 'longitude_usuario',
    'smoker', 'drink_level', 'drink_level_numerico',
    'dress_preference', 'dress_preference_numerico',
    'ambience', 'transport', 'marital_status', 'hijos',
    'birth_year', 'edad', 'grupo_edad',
    'interest', 'personality', 'religion', 'activity',
    'color', 'weight', 'height', 'budget', 'budget_numerico',
    'Rcuisine_user', 'Upayment', 'preferencia_cocina',
    'presupuesto_compatible', 'frecuencia_visitas_usuario',
    'afinidad_usuario_cocina'
]

In [82]:
restaurant_cols = [
    'placeID', 'name', 'address', 'city', 'state', 'country', 'zip',
    'latitude_rest', 'longitude_rest', 'the_geom_meter',
    'alcohol', 'smoking_area', 'dress_code', 'accessibility',
    'price', 'price_numerico', 'Rambience', 'franchise', 'area',
    'other_services', 'parking_lot', 'tiene_estacionamiento',
    'Rcuisine_rest', 'Rpayment', 'hours', 'days',
    'clientes_totales_restaurante', 'diversidad_cocina'
]

In [83]:
interaction_cols = [
    'user_count',
    'restaurant_count',
    'food_rating',
    'service_rating',
    'rating',
    'overall_success',
    'success_clase'
]

In [84]:
nuevo_orden = user_cols + restaurant_cols + interaction_cols

Comprobamos que todas las columnas que acabamos de listar, estén realmente en el dataframe original, para luego solo reordenarlas. 

In [85]:
missing_cols = [col for col in nuevo_orden if col not in df.columns]
print("Columnas que faltan en el DataFrame:", missing_cols)

Columnas que faltan en el DataFrame: []


In [86]:
df = df[nuevo_orden]

In [87]:
df.columns

Index(['userID', 'latitude_usuario', 'longitude_usuario', 'smoker',
       'drink_level', 'drink_level_numerico', 'dress_preference',
       'dress_preference_numerico', 'ambience', 'transport', 'marital_status',
       'hijos', 'birth_year', 'edad', 'grupo_edad', 'interest', 'personality',
       'religion', 'activity', 'color', 'weight', 'height', 'budget',
       'budget_numerico', 'Rcuisine_user', 'Upayment', 'preferencia_cocina',
       'presupuesto_compatible', 'frecuencia_visitas_usuario',
       'afinidad_usuario_cocina', 'placeID', 'name', 'address', 'city',
       'state', 'country', 'zip', 'latitude_rest', 'longitude_rest',
       'the_geom_meter', 'alcohol', 'smoking_area', 'dress_code',
       'accessibility', 'price', 'price_numerico', 'Rambience', 'franchise',
       'area', 'other_services', 'parking_lot', 'tiene_estacionamiento',
       'Rcuisine_rest', 'Rpayment', 'hours', 'days',
       'clientes_totales_restaurante', 'diversidad_cocina', 'user_count',
       'restau

In [88]:
df.shape

(28434, 65)

In [89]:
df.head()

,userID,latitude_usuario,longitude_usuario,smoker,drink_level,drink_level_numerico,dress_preference,dress_preference_numerico,ambience,transport,...,days,clientes_totales_restaurante,diversidad_cocina,user_count,restaurant_count,food_rating,service_rating,rating,overall_success,success_clase
0,U1077,22.156469,-100.98554,0,social drinker,3,elegant,4,family,public,...,Mon;Tue;Wed;Thu;Fri;,36,1,19986,40,2,2,2,1.0,alto
1,U1077,22.156469,-100.98554,0,social drinker,3,elegant,4,family,public,...,Mon;Tue;Wed;Thu;Fri;,36,1,19986,40,2,2,2,1.0,alto
2,U1077,22.156469,-100.98554,0,social drinker,3,elegant,4,family,public,...,Mon;Tue;Wed;Thu;Fri;,36,1,19986,40,2,2,2,1.0,alto
3,U1077,22.156469,-100.98554,0,social drinker,3,elegant,4,family,public,...,Sat;,36,1,19986,40,2,2,2,1.0,alto
4,U1077,22.156469,-100.98554,0,social drinker,3,elegant,4,family,public,...,Sat;,36,1,19986,40,2,2,2,1.0,alto


Ahora que tenemos todos los datos de San Luis Potosí ordenados correctamente, procedemos a reconstruir las tablas de restaurantes.csv y usuarios.csv, pero esta vez, unicamente restringidas a la unidad muestral seleccionada (ciudad de San Luis Potosí). 

In [90]:
# Variables deseadas para la tabla de restaurantes
restaurante_vars = [
    'placeID', 'latitude_rest', 'longitude_rest', 'the_geom_meter', 'name',
    'address', 'city', 'state', 'country', 'zip', 'alcohol',
    'smoking_area', 'dress_code', 'accessibility', 'price',
    'Rambience', 'franchise', 'area', 'other_services'
]

# Variables deseadas para la tabla de usuarios
usuario_vars = [
    'userID', 'latitude_usuario', 'longitude_usuario', 'smoker',
    'drink_level', 'dress_preference', 'ambience', 'transport',
    'marital_status', 'hijos', 'birth_year', 'interest', 'personality',
    'religion', 'activity', 'color', 'weight', 'budget', 'height'
]

# Crear tablas únicas
restaurantes = df[restaurante_vars].drop_duplicates(subset='placeID')
usuarios = df[usuario_vars].drop_duplicates(subset='userID')

In [91]:
# Guardar nuevas tablas en CSV
ruta_restaurantes = 'datos/restaurantes_slp.csv'
ruta_usuarios = 'datos/usuarios_slp.csv'

restaurantes.to_csv(ruta_restaurantes, index=False)
usuarios.to_csv(ruta_usuarios, index=False)

### 2.3 Remoción de Variables Unitarias

Procedemos a crear una función que nos permita ver las columnas unitarias en nuestro dataframe. 

In [92]:
def var_unit(df):
    for col in df.columns:
        if len(df[col].unique()) == 1:
            print(f"Variable Unitaria: {col}")
            print(f"Tipo de variable: {df[col].dtype}")
            print(f"Valor Único: {df[col].unique()}")
        else:
            pass

In [93]:
var_unit(df)

Variable Unitaria: city
Tipo de variable: object
Valor Único: ['San Luis Potosí']
Variable Unitaria: state
Tipo de variable: object
Valor Único: ['San Luis Potosí']
Variable Unitaria: country
Tipo de variable: object
Valor Único: ['México']


Para fines prácticos, removemos estas vaiables unitarias, pues no aportan información que no tengamos ya. 

In [94]:
df = df.drop(columns=['city','state','country'])

In [95]:
var_unit(df)

In [96]:
df.shape

(28434, 62)

Finalmente, exportamos esta tabla estructurada a csv para poder generar un reporte BI en Google Looker Studio. 

In [97]:
df.to_csv('tabla_analitica.csv')

El dashboard puede ser accesado dando click [aquí](https://lookerstudio.google.com/reporting/5c03e897-f3d2-44d6-8a12-f0f89bfb3a4c)

# 3. Tratamiento de Valores Extremos

Para tratar valores extremos, primero crearemos unas funciones que nos ayudarar a tratarlos de manera univariada (método IQR y Z-Score) y multivariada (método Isolation Forest y DBSCAN). Luego, creamos otra función que nos permita comparar la información restante que nos queda al tratar el dataframe con cada uno de los métodos. 

También definimos otra función que nos permite asegurarnos de que removamos las columnas que presenten correlación 1 en valor absoluto. 

In [98]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy import stats

# 1. UNIVARIADO: OUTLIERS POR IQR O Z-SCORE
def tratar_outliers_univariado(df, metodo='IQR'):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if metodo == 'IQR':
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            df = df[(df[col] >= lower) & (df[col] <= upper)]
    elif metodo == 'Z-Score':
        z_scores = np.abs(stats.zscore(df[numeric_cols]))
        df = df[(z_scores < 3).all(axis=1)]
    else:
        raise ValueError("Método no válido. Usa 'IQR' o 'Z-Score'.")
    return df

# 2. MULTIVARIADO: OUTLIERS POR ISOLATION FOREST O DBSCAN
def tratar_outliers_multivariado(df, metodo='IsolationForest'):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[numeric_cols])

    if metodo == 'IsolationForest':
        iso = IsolationForest(contamination=0.05, random_state=42)
        outliers = iso.fit_predict(X_scaled)
        df = df[outliers == 1]
    elif metodo == 'DBSCAN':
        db = DBSCAN(eps=1.5, min_samples=5)
        clusters = db.fit_predict(X_scaled)
        df = df[clusters != -1]
    else:
        raise ValueError("Método no válido. Usa 'IsolationForest' o 'DBSCAN'.")
    return df

# 3. COMPARACIÓN DE PÉRDIDA DE INFORMACIÓN
def comparar_metodos_outliers(df):
    resultados = {}
    total_original = df.shape[0]

    # Métodos univariados
    for metodo in ['IQR', 'Z-Score']:
        df_filtrado = tratar_outliers_univariado(df, metodo)
        resultados[f"Univariado - {metodo}"] = total_original - df_filtrado.shape[0]

    # Métodos multivariados
    for metodo in ['IsolationForest', 'DBSCAN']:
        df_filtrado = tratar_outliers_multivariado(df, metodo)
        resultados[f"Multivariado - {metodo}"] = total_original - df_filtrado.shape[0]

    resultados_df = pd.DataFrame.from_dict(resultados, orient='index', columns=['Filas eliminadas'])
    resultados_df['% pérdida'] = (resultados_df['Filas eliminadas'] / total_original * 100).round(2)
    return resultados_df

# 4. REMOVER VARIABLES CON CORRELACIÓN = 1 O -1
def remover_variables_correlacionadas(df):
    # Solo selecciona columnas numéricas
    df_numerico = df.select_dtypes(include=[np.number])
    
    # Calcular matriz de correlación
    corr_matrix = df_numerico.corr().abs()
    
    # Obtener triángulo superior para evitar duplicados
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Identificar columnas con correlación 1
    columnas_a_remover = [col for col in upper.columns if any(upper[col] == 1)]
    
    # Eliminar del dataframe original (no solo numérico)
    df_reducido = df.drop(columns=columnas_a_remover)
    
    return df_reducido, columnas_a_remover


In [99]:
# Comparar métodos
resumen = comparar_metodos_outliers(df)
print(resumen)

# Elegir método óptimo:
df_tratado = tratar_outliers_multivariado(df, metodo='IsolationForest')

# Remover variables altamente correlacionadas
df_final, columnas_removidas = remover_variables_correlacionadas(df_tratado)

print(f"Columnas removidas por correlación perfecta: {columnas_removidas}")


                                Filas eliminadas  % pérdida
Univariado - IQR                           22179      78.00
Univariado - Z-Score                       10527      37.02
Multivariado - IsolationForest              1158       4.07
Multivariado - DBSCAN                        762       2.68
Columnas removidas por correlación perfecta: []


Podemos notar que, el método que más nos conviene usar es el método multivariado de DBSCAN, pues con este metodo, solamente perdemos 2.68% de la información original.  

Además, también podemos notar que no se han encontrado columnas con correlación perfecta. Esto nos dice que, podemos continuar a remover multicolinealidad en columnas. 

# 4. Multicolinealidad y Reducción de Dimensiones

## 4.1 Multicolinealidad con VarClusHi 

Este método nos permitira eliminar aquellas columnas que presenten multicolinealidad con otras columnas. Esto nos ayudara a obtener un dataframe de dimensiones más reducidas, pero con variables no correlacionadas que puedan proporcionar información útil libre de sesgos por colinealidad. 

Definiremos una función que aplicara un método de clustering en nuestras variables a partir de VarClusHi. También haremos que nuestra función genere un gráfico interactivo sobre como las variables ahora están agrupadas por distintos clusters. 

In [100]:
import plotly.express as px
from varclushi import VarClusHi

def clustering_variables_varclushi(df, target_column='success_clase', threshold=1.0):
    df = df.copy()

    # Filtrar variables numéricas y eliminar la variable objetivo
    X_num = df.drop(columns=[target_column]).select_dtypes(include='number')
    X_num = X_num.loc[:, X_num.nunique() > 1].dropna(axis=1, how='any')

    # Aplicar VarClusHi
    clusterer = VarClusHi(X_num, maxeigval2=threshold)
    clusterer.varclus()  # realiza el clustering, guarda internamente

    # Obtener resultados
    cluster_df = clusterer.rsquare  # este es el DataFrame real con resultados

    # Seleccionar una variable representativa por clúster (mayor RS_Ratio)
    selected = cluster_df.sort_values(by='RS_Ratio', ascending=False).groupby('Cluster').head(1)
    columnas_seleccionadas = selected['Variable'].tolist()

    # Graficar con Plotly
    fig = px.scatter(
        cluster_df,
        x='RS_Ratio',
        y='Cluster',
        text='Variable',
        color='Cluster',
        color_continuous_scale='Viridis',
        title='Clustering de Variables con VarClusHi',
        template='plotly_dark'
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(
        xaxis_title='RS_Ratio (representatividad)',
        yaxis_title='Número de Clúster',
        showlegend=False
    )
    fig.show()

    # Retornar DataFrame reducido y resumen
    return df[columnas_seleccionadas + [target_column]], columnas_seleccionadas, cluster_df


In [101]:
df_reducido, columnas_seleccionadas, resumen_clustering = clustering_variables_varclushi(df_tratado)

Comprobamos que nuestro dataframe ahora tiene menos variables, pues las variables correlacionadas han sido eliminadas. 

In [102]:
df_reducido.shape

(27276, 11)

## 4.2 Reducción de Dimensiones con PCA 

Una vez que aseguramos que nuestro dataframe no contiene variables correlacionadas, lo que haremos será reducir aún mas nuestro dataframe a solo dos variables, llamadas componentes principales. Para ello, utilizaremos el escalador estandar, pues este nos permite normalizar los datos y con ello, el PCA podrá formár las nuevas componentes principales como combinaciones lineales de las variables que ahora tenemos, justo después del clustering.  

In [103]:
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def aplicar_pca_visualizacion_plotly(df, target_column='success_clase'):
    df = df.copy()

    y = df[target_column]
    X = df.drop(columns=[target_column])

    X_num = X.select_dtypes(include='number')

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_num)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    df_pca = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])
    df_pca[target_column] = y.values

    fig = px.scatter(
        df_pca, x='PC1', y='PC2', color=target_column,
        title="PCA (2D) - Clases de Éxito",
        template='plotly_dark',
        color_discrete_sequence=px.colors.qualitative.Plotly
    )
    fig.update_traces(marker=dict(size=8, opacity=0.7), selector=dict(mode='markers'))
    fig.show()

    return df_pca


Ahora, aplicamos las funcion que nos permitira ver las nuevas componentes principales 2D, pero agrendo también la variable objetivo, para ver como estos puntos en el plano se distribuyen según la variable objetivo. 

In [104]:
aplicar_pca_visualizacion_plotly(df_reducido)

,PC1,PC2,success_clase
0,-0.956563,1.212321,alto
1,-0.956563,1.212321,alto
2,-0.956563,1.212321,alto
3,-0.956563,1.212321,alto
4,-0.956563,1.212321,alto
...,...,...,...
27271,-0.697622,0.088558,alto
27272,-0.697622,0.088558,alto
27273,-0.697622,0.088558,alto
27274,-0.697622,0.088558,alto


# 5. Selección de Variables y Transformación Entropica 

## 5.1 Poder Predictivo con SelectKBest 

El método de SelectKBest nos permite seleccionar las mejores variables con potencial predictivo utilizando un test estadístico respecto a la variable objetivo. 

Esta función se asegura de tomar exclusivamente variables numéricas y con ello seleccionar las mejores variables para prepararlas para un modelo predictivo. 

In [105]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import LabelEncoder

def seleccionar_variables_kbest(df, target_column='success_clase', k=10):
    df = df.copy()

    # Codificar clases en números
    y = LabelEncoder().fit_transform(df[target_column])
    X = df.drop(columns=[target_column])
    X_num = X.select_dtypes(include=[np.number])

    # Si tienes valores negativos, usa f_classif en vez de chi2
    if (X_num < 0).any().any():
        from sklearn.feature_selection import f_classif
        selector = SelectKBest(score_func=f_classif, k=k)
    else:
        selector = SelectKBest(score_func=chi2, k=k)

    selector.fit(X_num, y)
    mask = selector.get_support()
    selected_features = X_num.columns[mask]

    return df[selected_features.tolist() + [target_column]], selected_features.tolist()


In [114]:
df_kbest = seleccionar_variables_kbest(df_tratado)

In [116]:
df_kbest

(       frecuencia_visitas_usuario  latitude_rest  longitude_rest  franchise  \
 0                              72      22.150802     -100.982680          0   
 1                              72      22.150802     -100.982680          0   
 2                              72      22.150802     -100.982680          0   
 3                              72      22.150802     -100.982680          0   
 4                              72      22.150802     -100.982680          0   
 ...                           ...            ...             ...        ...   
 28429                          90      22.156883     -100.978485          0   
 28430                          90      22.156883     -100.978485          0   
 28431                          90      22.156883     -100.978485          0   
 28432                          90      22.156883     -100.978485          0   
 28433                          90      22.156883     -100.978485          0   
 
        area  clientes_totales_restaur

Una vez que hemos trabajado todas las tecnicas de reducción de dimensionalidad y poder predictivo, podemos proceder a comparar los métodos de Clustering y SelectKbest, y revisar que columnas conservaron o elimnaron cada uno. 

In [109]:
def comparar_tecnicas_seleccion(df, target_column='success_clase', k_best=10, clustering_threshold=0.9):
    resultados = {}

    # 1. Clustering con VarClusHi
    df_clust, var_clust, resumen_clust = clustering_variables_varclushi(df, threshold=clustering_threshold, target_column=target_column)
    resultados['Clustering'] = var_clust

    # 2. SelectKBest
    df_kbest, var_kbest = seleccionar_variables_kbest(df, target_column=target_column, k=k_best)
    resultados['SelectKBest'] = var_kbest

    # Combinar en tabla resumen
    todas_vars = sorted(set(sum(resultados.values(), [])))
    resumen = pd.DataFrame(index=todas_vars)

    for metodo, variables in resultados.items():
        resumen[metodo] = resumen.index.isin(variables).astype(int)

    resumen['Total Seleccionado'] = resumen.sum(axis=1)
    resumen = resumen.sort_values(by='Total Seleccionado', ascending=False)

    return resumen

In [111]:
resumen_comparacion = comparar_tecnicas_seleccion(df_tratado)

In [113]:
print(resumen_comparacion)

                              Clustering  SelectKBest  Total Seleccionado
frecuencia_visitas_usuario             1            1                   2
diversidad_cocina                      1            1                   2
restaurant_count                       1            1                   2
latitude_rest                          1            1                   2
afinidad_usuario_cocina                1            0                   1
Rambience                              1            0                   1
dress_preference_numerico              1            0                   1
drink_level_numerico                   1            0                   1
clientes_totales_restaurante           0            1                   1
area                                   0            1                   1
height                                 1            0                   1
franchise                              0            1                   1
overall_success                       

## 5.2 Transformación Entropica con IV y WoE 

En este caso, dado que nuestra variabe objetivo es una variable discreta de tres posibles valores, los metodos WoE y IV no aplican, pues estos métodos están principalmente pensados para trabajar con variables objetivo binarias, pues ambas tratan de definir un rango de proporcion para "eventos" y "no eventos", en nuestro espacio muestral. 

# 6. Conclusiones

Podemos notar que, la población de usuarios de restaurantes en San Luis Potosí es muy diversa. Observamos que hay todo tipo de preferencias y perfiles de usuarios. 

Una vez, concluida toda la ingenieria de datos, se puede comprobar que existen ciertas variables que juegan un papel fundamental para medir el éxito de un restaurante. Se pueden mencionar algunas de ellas, como son las siguientes variables 


* `diversidad_cocina`: Mientras más variedad de cocina tenga un restaurante, más tendrá afinidad con una alta gama de clientes      
* `frecuencia_visitas_usuario`: Mide la preferencia de los usuarios
* `restaurant_count`: Miden la popularidad del restaurante.                                                             |


Podemos también notar que estas variables fueron elegidas tanto en el método de Clustering y Select-KBest 
```
Clustering SelectKBest Total_Seleccionado 
frecuencia_visitas_usuario 1 1 2 
diversidad_cocina 1 1 2 
restaurant_count 1 1 2
```

Esto también tiene sentido si revisamos los datos que tenemos en nuestro [dashboard](https://lookerstudio.google.com/reporting/5c03e897-f3d2-44d6-8a12-f0f89bfb3a4c). Podemos ver que, los restaurantes que tuvieron más éxito no necesariamente fueron los restaurantes que tuvieron mejores rating (la mayoría de los ratings no fue muy bueno). En realidad, influyen otro tipo de cosas, como el hecho de que, el púbico potosino interactua distinto con negocios locales y franquicias, pero sobre todo el hecho de que el restaurante se alinie con el mayor número de preferencias de los usuarios. 